# 03 — Score postings and export the shared data model

This notebook turns the Silver working file into explainable scores, rule flags, and the 12 CSV tables used by Excel, MySQL, and Power BI.

In [ ]:
# Import libraries, settings, and the output from notebook 02.
from pathlib import Path
import json
import re
import pandas as pd
import yaml


def find_project_root(start: Path) -> Path:
    """Find the repository root without relying on a fixed Jupyter folder."""
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks" / "data").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project folder.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
with (PROJECT_ROOT / "notebooks" / "analysis_settings.yaml").open(encoding="utf-8") as file:
    settings = yaml.safe_load(file)

SILVER_PATH = PROJECT_ROOT / "notebooks" / "data" / "working" / "02_silver_postings.csv"
EXPORT_DIR = PROJECT_ROOT / "notebooks" / "data" / "exports"
SUMMARY_PATH = PROJECT_ROOT / "notebooks" / "data" / "qa_summary.json"
if not SILVER_PATH.exists():
    raise FileNotFoundError("Run notebook 02 before this notebook.")

silver = pd.read_csv(SILVER_PATH, low_memory=False, dtype={"source_job_id": "string"})
print(f"Silver rows loaded: {len(silver):,}")

In [ ]:
# Define the visible role and skill rules used by this portfolio project.
ROLE_RULES = {
    "Data Analyst": {
        "role_family": "Analytics",
        "occupational_group": "Data & BI",
        "keywords": ["data analyst", "reporting analyst", "analytics analyst"],
        "expected_skills": ["SQL", "Excel", "Power BI", "Python"]
    },
    "Business Analyst": {
        "role_family": "Analytics",
        "occupational_group": "Strategy & Operations",
        "keywords": ["business analyst", "functional analyst", "systems analyst"],
        "expected_skills": ["Excel", "SQL", "Power BI"]
    },
    "BI Analyst / BI Developer": {
        "role_family": "Analytics",
        "occupational_group": "Data & BI",
        "keywords": ["bi analyst", "bi developer", "business intelligence", "power bi developer"],
        "expected_skills": ["Power BI", "SQL", "DAX"]
    },
    "Data Engineer": {
        "role_family": "Data Engineering",
        "occupational_group": "Engineering",
        "keywords": ["data engineer", "etl developer", "data warehouse engineer"],
        "expected_skills": ["Python", "SQL", "ETL", "Spark"]
    },
    "Software Engineer": {
        "role_family": "Software Engineering",
        "occupational_group": "Engineering",
        "keywords": ["software engineer", "software developer", "backend engineer", "frontend engineer"],
        "expected_skills": ["Python", "SQL", "Java", "JavaScript"]
    }
}

SKILL_RULES = {
    "SQL": ["sql", "structured query language"],
    "Excel": ["excel", "spreadsheets", "vlookup", "pivot table"],
    "Power BI": ["power bi", "powerbi", "dax", "power query"],
    "Python": ["python", "pandas", "numpy"],
    "ETL": ["etl", "data pipeline", "data warehousing"],
    "Spark": ["spark", "pyspark"],
    "Java": ["java", "jvm"],
    "JavaScript": ["javascript", "typescript", "node.js"],
    "Tableau": ["tableau"],
    "AWS": ["aws", "amazon web services"],
    "Azure": ["azure", "synapse"],
    "Git": ["git", "github", "gitlab"]
}

SKILL_CATEGORY = {
    "SQL": "Query Languages", "Excel": "Spreadsheet", "Power BI": "BI / Visualization",
    "Python": "Programming", "ETL": "Data Engineering", "Spark": "Data Engineering",
    "Java": "Programming", "JavaScript": "Programming", "Tableau": "BI / Visualization",
    "AWS": "Cloud", "Azure": "Cloud", "Git": "Data Engineering"
}

In [ ]:
# Classify roles and detect skills with small functions you can change and rerun.
def classify_role(title: str) -> dict:
    """Return a named role and its business grouping, or an explicit unclassified value."""
    title = "" if pd.isna(title) else str(title).lower()
    for role_name, rule in ROLE_RULES.items():
        if any(keyword in title for keyword in rule["keywords"]):
            return {
                "canonical_role_name": role_name,
                "role_family": rule["role_family"],
                "occupational_group": rule["occupational_group"]
            }
    return {
        "canonical_role_name": "Unclassified",
        "role_family": "Other",
        "occupational_group": "Other"
    }


def extract_skills(text: str) -> list[str]:
    """Find recognised skills with word boundaries to avoid partial-word matches."""
    text = "" if pd.isna(text) else str(text).lower()
    found = []
    for skill_name, aliases in SKILL_RULES.items():
        if any(re.search(r"(?<!\w)" + re.escape(alias) + r"(?!\w)", text) for alias in aliases):
            found.append(skill_name)
    return found


role_details = silver["title_normalized"].map(classify_role)
silver[["canonical_role_name", "role_family", "occupational_group"]] = pd.DataFrame(
    role_details.tolist(), index=silver.index
)
silver["skill_text"] = silver["description_clean"].fillna("") + " " + silver["skills_raw"].fillna("")
silver["detected_skills"] = silver["skill_text"].map(extract_skills)
silver[["title_raw", "canonical_role_name", "detected_skills"]].head(10)

In [ ]:
# Calculate each score component separately so the final score stays explainable.
expected_skill_map = {name: rule["expected_skills"] for name, rule in ROLE_RULES.items()}

# Completeness checks six candidate-facing fields; each populated field contributes equally.
completeness_fields = pd.DataFrame({
    "title": silver["title_normalized"].fillna("").ne(""),
    "company": silver["company_normalized"].fillna("").ne(""),
    "description": silver["description_clean"].fillna("").ne(""),
    "location": silver["location_normalized"].fillna("").ne(""),
    "workplace": silver["workplace_type"].ne("unknown"),
    "salary": silver["salary_numeric_present"].fillna(False)
})
silver["completeness_score"] = completeness_fields.sum(axis=1).div(len(completeness_fields.columns)).mul(100)

# Penalise only an explicit entry-level title that asks for two or more years.
silver["experience_consistency_score"] = 100.0
entry_two_plus = (silver["seniority_title"] == "entry") & (silver["experience_min_years"].fillna(-1) >= 2)
entry_five_plus = (silver["seniority_title"] == "entry") & (silver["experience_min_years"].fillna(-1) >= 5)
silver.loc[entry_two_plus, "experience_consistency_score"] = 70.0
silver.loc[entry_five_plus, "experience_consistency_score"] = 50.0

# Score the other transparent signals with simple, inspectable rules.
silver["salary_transparency_score"] = silver["salary_numeric_present"].fillna(False).map({True: 100.0, False: 0.0})
silver["location_clarity_score"] = 0.0
silver.loc[silver["location_normalized"].fillna("").ne(""), "location_clarity_score"] = 60.0
silver.loc[silver["workplace_type"].ne("unknown"), "location_clarity_score"] += 40.0


def role_alignment(row) -> float:
    """Reward a recognised role when at least one expected skill appears in the posting text."""
    expected = expected_skill_map.get(row["canonical_role_name"], [])
    if not expected:
        return 40.0
    return 100.0 if set(expected).intersection(row["detected_skills"]) else 50.0


silver["role_alignment_score"] = silver.apply(role_alignment, axis=1)
silver["duplicate_repost_score"] = silver["duplicate_cluster_size"].eq(1).map({True: 100.0, False: 40.0})

In [ ]:
# Combine the components with the weights stored in the editable settings file.
weights = settings["scoring"]["weights"]
if any(value < 0 for value in weights.values()) or abs(sum(weights.values()) - 1.0) > 1e-9:
    raise ValueError("Weights must be non-negative and add up to 1.")
weighted_components = {
    "completeness": "completeness_score",
    "experience_consistency": "experience_consistency_score",
    "salary_transparency": "salary_transparency_score",
    "location_clarity": "location_clarity_score",
    "role_alignment": "role_alignment_score",
    "duplicate_repost": "duplicate_repost_score"
}
silver["jd_quality_score"] = sum(
    silver[column] * weights[weight_name]
    for weight_name, column in weighted_components.items()
).round(2)


def quality_band(score: float) -> str:
    """Turn a numeric score into one reporting label with visible thresholds."""
    bands = settings["scoring"]["quality_bands"]
    if score >= bands["high_quality"]:
        return "High Quality / Clear"
    if score >= bands["generally_clear"]:
        return "Generally Clear"
    if score >= bands["review_recommended"]:
        return "Review Recommended"
    return "Multiple Quality Concerns"


silver["quality_band"] = silver["jd_quality_score"].map(quality_band)
silver[["jd_quality_score", "quality_band"]].describe(include="all")

In [ ]:
# Build dimension tables with stable integer keys for the shared star schema.
silver["company_display"] = silver["company_raw"].fillna("Unknown").replace("", "Unknown")
silver["company_normalized"] = silver["company_normalized"].fillna("unknown").replace("", "unknown")
silver["location_display"] = silver["location_raw"].fillna("Unknown").replace("", "Unknown")
silver["location_normalized"] = silver["location_normalized"].fillna("unknown").replace("", "unknown")

# Each dimension is created once, then its key is mapped back to every posting.
dim_company = silver[["company_display", "company_normalized"]].drop_duplicates("company_normalized").sort_values("company_normalized").reset_index(drop=True)
dim_company["company_key"] = dim_company.index + 1
dim_company = dim_company.rename(columns={"company_display": "company_name", "company_normalized": "normalized_company_name"})
dim_company["industry"] = pd.NA
dim_company["country"] = "Unknown"
company_map = dict(zip(dim_company["normalized_company_name"], dim_company["company_key"]))

role_columns = ["canonical_role_name", "role_family", "occupational_group"]
dim_role = silver[role_columns].drop_duplicates().sort_values("canonical_role_name").reset_index(drop=True)
dim_role["role_key"] = dim_role.index + 1
role_map = dict(zip(dim_role["canonical_role_name"], dim_role["role_key"]))

location_columns = ["location_display", "location_normalized", "workplace_type"]
dim_location = silver[location_columns].drop_duplicates(["location_normalized", "workplace_type"]).sort_values(location_columns).reset_index(drop=True)
dim_location["location_key"] = dim_location.index + 1
dim_location["city"] = dim_location["location_display"]  # Source label, not a verified city.
dim_location["region"] = pd.NA
dim_location["country"] = "Unknown"
dim_location["metro"] = dim_location["city"].fillna("Unknown")
dim_location = dim_location.rename(columns={"location_normalized": "location_normalized"})
location_map = dict(zip(zip(dim_location["location_normalized"], dim_location["workplace_type"]), dim_location["location_key"]))

source_name = settings["source"]["name"]
dim_source = pd.DataFrame([{
    "source_key": 1,
    "source_name": source_name,
    "source_type": "historical_file",
    "platform_or_ats": "LinkedIn historical snapshot",
    "collection_method": "retained_historical_snapshot"
}])

In [ ]:
# Add job and date keys, then build the central posting fact table.
silver["company_key"] = silver["company_normalized"].map(company_map)
silver["role_key"] = silver["canonical_role_name"].map(role_map)
silver["location_key"] = [
    location_map[(location, workplace)]
    for location, workplace in zip(silver["location_normalized"], silver["workplace_type"])
]

# One canonical key creates one dim_job record even when an exact repost appears more than once.
dim_job = silver[["canonical_job_key", "title_normalized", "company_key", "role_key", "location_key"]].drop_duplicates("canonical_job_key").sort_values("canonical_job_key").reset_index(drop=True)
dim_job["job_key"] = dim_job.index + 1
job_map = dict(zip(dim_job["canonical_job_key"], dim_job["job_key"]))
silver["job_key"] = silver["canonical_job_key"].map(job_map)

snapshot_date = pd.Timestamp(settings["source"]["snapshot_date"])
published_dates = pd.to_datetime(silver["listed_at"], utc=True, errors="coerce").dt.tz_localize(None).dt.normalize()
known_dates = pd.concat([published_dates.dropna(), pd.Series([snapshot_date])])
all_dates = pd.date_range(known_dates.min(), known_dates.max(), freq="D")
dim_date = pd.DataFrame({"full_date": all_dates})
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["quarter"] = dim_date["full_date"].dt.quarter
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["month_name"] = dim_date["full_date"].dt.month_name()
dim_date["day"] = dim_date["full_date"].dt.day
dim_date["day_of_week"] = dim_date["full_date"].dt.day_name()
dim_date["is_weekend"] = dim_date["full_date"].dt.dayofweek.ge(5).astype(int)
date_map = dict(zip(dim_date["full_date"].dt.date, dim_date["date_key"]))

silver["published_date_key"] = published_dates.dt.date.map(date_map).astype("Int64")
silver["observed_date_key"] = date_map[snapshot_date.date()]
silver["posting_observation_key"] = range(1, len(silver) + 1)

fact_job_posting = silver[[
    "posting_observation_key", "job_key", "company_key", "role_key", "location_key",
    "published_date_key", "observed_date_key", "source_job_id", "title_raw", "title_normalized",
    "description_clean", "seniority_title", "employment_type", "workplace_type",
    "experience_min_years", "experience_max_years", "salary_min", "salary_max", "salary_median", "currency",
    "pay_period", "posting_age_days", "description_word_count", "salary_numeric_present",
    "experience_numeric_present", "exact_duplicate_key"
]].copy()
fact_job_posting.insert(2, "source_key", 1)
fact_job_posting = fact_job_posting.rename(columns={"currency": "salary_currency", "pay_period": "salary_period", "exact_duplicate_key": "payload_hash"})

In [ ]:
# Create row-level flags, job-quality facts, skills, and exact repost events.
quality_base = silver.sort_values(["listed_at", "source_job_id"], na_position="first").drop_duplicates("job_key", keep="last").copy()
# Use the latest observed posting for one job-level score and one set of flags.
flag_rows = []
for row in quality_base.itertuples(index=False):
    if row.seniority_title == "entry" and pd.notna(row.experience_min_years) and row.experience_min_years >= 2:
        flag_rows.append({"job_key": row.job_key, "rule_code": "ENTRY_ROLE_REQUIRES_2PLUS_YEARS", "severity": "medium", "observed_value": f"{row.experience_min_years} years", "expected_value": "Less than 2 years for entry-level review", "evidence_text": "Entry-level title paired with an explicit two-or-more-years requirement."})
    if not row.salary_numeric_present:
        flag_rows.append({"job_key": row.job_key, "rule_code": "NUMERIC_SALARY_NOT_DISCLOSED", "severity": "low", "observed_value": "No numeric salary", "expected_value": "Numeric salary range", "evidence_text": "Posting does not include a numeric salary value."})
    if row.duplicate_cluster_size > 1:
        flag_rows.append({"job_key": row.job_key, "rule_code": "EXACT_REPOST_CLUSTER", "severity": "low", "observed_value": f"Cluster size {row.duplicate_cluster_size}", "expected_value": "One exact record", "evidence_text": "Exact company, title, location, and description combination repeats."})

fact_job_flag = pd.DataFrame(flag_rows)
if fact_job_flag.empty:
    fact_job_flag = pd.DataFrame(columns=["job_key", "rule_code", "severity", "observed_value", "expected_value", "evidence_text"])
fact_job_flag.insert(0, "flag_id", range(1, len(fact_job_flag) + 1))
fact_job_flag["created_at"] = snapshot_date.strftime("%Y-%m-%d %H:%M:%S")

flag_count = fact_job_flag.groupby("job_key").size() if not fact_job_flag.empty else pd.Series(dtype="int64")
fact_job_quality = quality_base[[
    "job_key", "jd_quality_score", "quality_band", "completeness_score", "experience_consistency_score",
    "role_alignment_score", "salary_transparency_score", "duplicate_repost_score", "location_clarity_score"
]].copy()
fact_job_quality.insert(1, "score_version", settings["run"]["score_version"])
fact_job_quality["flag_count"] = fact_job_quality["job_key"].map(flag_count).fillna(0).astype(int)
fact_job_quality["scored_at"] = snapshot_date.strftime("%Y-%m-%d %H:%M:%S")

skill_rows = []
for row in quality_base[["job_key", "detected_skills"]].itertuples(index=False):
    for skill in row.detected_skills:
        skill_rows.append({"job_key": row.job_key, "skill_name": skill})
skill_links = pd.DataFrame(skill_rows).drop_duplicates() if skill_rows else pd.DataFrame(columns=["job_key", "skill_name"])
dim_skill = pd.DataFrame({"skill_name": sorted(skill_links["skill_name"].unique())})
dim_skill["skill_key"] = dim_skill.index + 1
dim_skill["category"] = dim_skill["skill_name"].map(SKILL_CATEGORY)
skill_map = dict(zip(dim_skill["skill_name"], dim_skill["skill_key"]))
bridge_job_skill = skill_links.assign(skill_key=skill_links["skill_name"].map(skill_map))[["job_key", "skill_key"]]

repost_base = silver[silver["duplicate_cluster_size"] > 1].sort_values(["canonical_job_key", "listed_at", "source_job_id"]).copy()
repost_base["previous_source_job_id"] = repost_base.groupby("canonical_job_key")["source_job_id"].shift()
repost_base = repost_base[repost_base["previous_source_job_id"].notna()].copy()
fact_repost_event = repost_base[["job_key", "previous_source_job_id", "source_job_id"]].rename(columns={"source_job_id": "current_source_job_id"})
fact_repost_event.insert(0, "repost_event_key", range(1, len(fact_repost_event) + 1))


In [ ]:
# Write all shared tables once so Excel, MySQL, and Power BI consume identical data.
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

tables = {
    "bridge_job_skill": bridge_job_skill,
    "dim_company": dim_company[["company_key", "company_name", "normalized_company_name", "industry", "country"]],
    "dim_date": dim_date[["date_key", "full_date", "year", "quarter", "month", "month_name", "day", "day_of_week", "is_weekend"]],
    "dim_job": dim_job[["job_key", "canonical_job_key", "title_normalized", "company_key", "role_key", "location_key"]],
    "dim_location": dim_location[["location_key", "city", "region", "country", "metro", "workplace_type", "location_normalized"]],
    "dim_role": dim_role[["role_key", "canonical_role_name", "role_family", "occupational_group"]],
    "dim_skill": dim_skill[["skill_key", "skill_name", "category"]],
    "dim_source": dim_source,
    "fact_job_flag": fact_job_flag,
    "fact_job_posting": fact_job_posting,
    "fact_job_quality": fact_job_quality,
    "fact_repost_event": fact_repost_event
}

# Validate every key and relationship before writing exports.
primary_keys = {
    "dim_company": ["company_key"], "dim_date": ["date_key"], "dim_job": ["job_key"],
    "dim_location": ["location_key"], "dim_role": ["role_key"], "dim_skill": ["skill_key"],
    "dim_source": ["source_key"], "fact_job_posting": ["posting_observation_key"],
    "fact_job_quality": ["job_key", "score_version"], "fact_job_flag": ["flag_id"],
    "bridge_job_skill": ["job_key", "skill_key"], "fact_repost_event": ["repost_event_key"]
}
for name, key in primary_keys.items():
    assert not tables[name].duplicated(key).any(), f"Duplicate key in {name}"
    assert not tables[name][key].isna().any().any(), f"Blank key in {name}"
relationships = [
    ("dim_job", "company_key", "dim_company", "company_key"),
    ("dim_job", "role_key", "dim_role", "role_key"),
    ("dim_job", "location_key", "dim_location", "location_key"),
    ("fact_job_posting", "source_key", "dim_source", "source_key"),
    ("fact_job_posting", "published_date_key", "dim_date", "date_key"),
    ("fact_job_posting", "observed_date_key", "dim_date", "date_key"),
    ("bridge_job_skill", "skill_key", "dim_skill", "skill_key"),
] + [(name, "job_key", "dim_job", "job_key") for name in
     ["fact_job_posting", "fact_job_quality", "fact_job_flag", "bridge_job_skill", "fact_repost_event"]]
for child, column, parent, parent_column in relationships:
    assert tables[child][column].dropna().isin(tables[parent][parent_column]).all(), f"Orphan: {child}.{column}"
assert fact_job_quality["jd_quality_score"].between(0, 100).all()
assert len(fact_job_quality) == len(dim_job)
assert len(fact_repost_event) == len(silver) - len(dim_job)
assert len(fact_job_posting) == len(silver)

for table_name, table in tables.items():
    table.to_csv(EXPORT_DIR / f"{table_name}.csv", index=False, na_rep="")

summary = {
    "rows_raw": int(len(pd.read_csv(PROJECT_ROOT / "notebooks" / "data" / "working" / "01_bronze_postings.csv"))),
    "rows_silver": int(len(silver)),
    "distinct_jobs": int(dim_job["job_key"].nunique()),
    "experience_numeric_coverage_pct": round(float(silver["experience_numeric_present"].mean() * 100), 2),
    "salary_numeric_coverage_pct": round(float(silver["salary_numeric_present"].mean() * 100), 2),
    "entry_jobs_count": int(silver.loc[silver["seniority_title"] == "entry", "job_key"].nunique()),
    "entry_jobs_requiring_2plus_years": int(silver.loc[entry_two_plus, "job_key"].nunique()),
    "entry_postings_count": int((silver["seniority_title"] == "entry").sum()),
    "entry_postings_requiring_2plus_years": int(entry_two_plus.sum()),
    "average_jd_quality": round(float(fact_job_quality["jd_quality_score"].mean()), 2),
    "quality_review_jobs": int(fact_job_quality["jd_quality_score"].lt(settings["scoring"]["quality_bands"]["generally_clear"]).sum()),
    "source_path": settings["source"]["path"],
    "row_limit": settings["run"]["row_limit"],
    "snapshot_date": str(settings["source"]["snapshot_date"]),
    "repost_events_count": int(len(fact_repost_event)),
    "total_flags_generated": int(len(fact_job_flag)),
    "score_version": settings["run"]["score_version"]
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Created {len(tables)} CSV files in {EXPORT_DIR.relative_to(PROJECT_ROOT)}")
pd.DataFrame([{"table": name, "rows": len(table)} for name, table in tables.items()])

In [ ]:
# Run small reconciliation checks before handing the files to another tool.
assert len(fact_job_posting) == len(silver), "Every Silver record must become one posting observation."
assert fact_job_posting["job_key"].isin(dim_job["job_key"]).all(), "Every posting must have a valid job key."
assert fact_job_quality["job_key"].isin(dim_job["job_key"]).all(), "Every quality score must have a valid job key."
assert len(set(tables).intersection({"fact_job_posting", "dim_job", "fact_job_quality"})) == 3, "Core export tables are missing."

quality_summary = (
    fact_job_quality.groupby("quality_band", dropna=False)
    .agg(jobs=("job_key", "size"), average_score=("jd_quality_score", "mean"))
    .round(2)
    .sort_values("jobs", ascending=False)
)
print("Reconciliation checks passed.")
quality_summary

## Your checkpoint
Explain one score calculation, the difference between jobs and postings, and one limitation before building the report.